# HDGfx Infector
> A polymorphic Windows file infector and USB worm built in 2026

- toc: true 
- badges: true
- categories: [hdgfx,fileinfector,worm,usb,polymorphic,config]

## Overview

HDGfx Infector is a Windows x86 file infector and USB worm built in mid-2026. As of 2026-11-17 there appears to be no public reporting on it despite its widespread propagation via sandboxes, specificaly the VirusTotal "[sandbox centipede](https://blog.unpac.me/2024/04/25/zombieware/)".

The malware is a small MinGW-w64 (GCC 8.1.0) DLL. Its export directory names it `System.dll`, and its export names describe what it does: `InfectFile`, `DisinfectFile`, `MetamorphicStub`, `GeneratePolymorphicAutorunInf` and so on. It infects 32-bit EXEs in place:

- A junk-filled entry stub overwrites the host entry point.
- A layer decryptor and an encrypted loader are added to the end of the host's last section.
- The loader decrypts the UPX-packed core DLL with RC4, maps it in memory, puts back the original entry bytes and returns to the host.

Every infection gets new keys and newly generated junk code. Once running, the core:

- infects EXEs on every fixed and removable drive;
- copies itself to each drive root as a hidden `.exe` or `.pif` with a locked, randomised `autorun.inf`;
- polls a No-IP dynamic-DNS host on TCP 8000 to download and run an update.

### Key points

- **Active development.** There are two builds about six weeks apart (version banners 15/07/2026 and 23/08/2026). The second adds a second update host and a new payload cipher.
- **Infects trusted software.** Infected files include Adobe, Java, Firefox and Google Update components. Infection removes their Authenticode signatures.
- **Targets AV installers.** It breaks files whose names contain `bitdef` or `kasp`.
- **Noisy.** Each infected program plays a sequence of beeps when it starts.
- **Still spreading.** The earliest sample we have was first seen on 2026-10-22. New infected files were still appearing on 2026-11-17.

### Name

"HDGfx Infector" is our working name. The malware has no family name, PDB path, copyright or version resource. The name comes from the only product-like string in the code, `Intel(R) HD Graphics Manager.exe`. This is the file name the worm asks its update server for.

The identifiers the malware does carry:

| Identifier | Value |
|---|---|
| DLL export name | `System.dll` |
| Version banner v1 | `======================= VERSION 15/07/2026 ======================` |
| Version banner v2 | `======================= VERSION 23/08/2026 ======================` |
| Mutex (v1 only) | `@0826` |
| Compiler | `GCC: (i686-posix-dwarf-rev0, Built by MinGW-W64 project) 8.1.0` |

### Timeline

| Build | PE timestamp | Banner date | Infection marker (dword) |
|---|---|---|---|
| v1 | 2026-07-24 05:48:59 | 15/07/2026 | `0x20260715` |
| v2 | 2026-08-26 20:42:41 | 23/08/2026 | `0x20260823` |

- The infection marker is the banner date as BCD.
- In both builds the banner date is a few days before the PE timestamp.
- The author controls all of these values, so they are consistent but not proof of the build dates.

v1 treats any marker at or below `0x20260714` as an old version and re-infects the file. That suggests builds older than 15 July 2026 exist, but we have not found one.

## Samples

Every infected host and both memory images below decrypt to the same v1 core, `ec55c4ba2ab0b530c4d613e9c0214611c734fc0d17e2fabec68ade33b980075c`.

### Core DLL

| SHA256 | Description |
|---|---|
| `ec55c4ba2ab0b530c4d613e9c0214611c734fc0d17e2fabec68ade33b980075c` | v1 core, UPX 4.24 packed (the encrypted payload in every host) |
| `80b49269fa01f64e34960f99922f93c53e0370d94c3158dac450281defb4d0bf` | v1 core, unpacked |
| `fbcbcd9c743695119f822da7d57e7f4605c93513b8cd70ed598d2aa942363af2` | v1 core, sandbox memory dump |
| `967c2c5bd65e8dc3fd46007e9ef19b5c117a5335d83080d0e9ffe61cf0d94041` | v1 core, sandbox memory dump |
| `0678e636166df303548c8c2f4d75611304f7f01d5881ac6e4baad16dae0aa566` | v2 core, memory dump (relocated to base `0x036A0000`) |

### Infected hosts

The 25 hosts come from 16 different programs.

| SHA256 | Host program |
|---|---|
| `a0c42a23fbcd3cd8c21db68517e18b5c229e9a2d95af62a5a57407ff92e48141` | Adobe Systems 1.824.35.0289 |
| `b448a0be6e7c3c25905bf6c97ecda197d3ef8f4f362a8920c8974c3c32cfe354` | Adobe Systems 1.824.35.0289 |
| `44a58d2e7d0f114939e067b1645fcc171ce60ffaa8ee918b7a35f2acd44a4945` | Adobe Systems 1.824.35.0289 |
| `5d90689c226a121eeca04ab5676430ed608a6d0f0c1339fb1684497346de9f82` | Adobe Systems 1.824.35.0289 |
| `7b232565f9697568eb4cc89dc49bf956ba2ee401a820dce93f28b6af704e8fd6` | AdobeARM.exe 1.824.35.0289 |
| `3cd13ce7ff46c49466c5484c0d7c08be4d97944288d356f2e1472dfa8c606e28` | AdobeARM.exe 1.824.35.0289 |
| `8015cea2882398190b58093a0715c5b21bd53510e7186c3db7f36e78cdacc349` | Adobe Inc. 1.824.460.1032 |
| `609372a70bd5b35befcf8e99a83355a049618f8cd9e6d53102a17811abfaa123` | Adobe Inc. 1.824.460.1032 |
| `cd551a44e7673c9a4a2007a7574ac1b1be7e439a3705461d5de07af70f90310c` | Adobe Setup.exe 20.6.20034.366983 |
| `59163fa30be47fbcce61171774d81f0283611ef51d2100561f501b3180dd8802` | Adobe Setup.exe 20.6.20034.366983 |
| `727966a9ec7f46a0227dcaf39be22193f24f2650e48541013bc90e3456286f90` | Adobe Setup.exe 20.6.20034.366983 |
| `249ea8bf7b6a76694253970e9312ee88d3e2144fe56479cfcad19966f827e834` | Adobe Setup.exe 18.9.20044.251705 |
| `8dd3e8a7ce2a7e4d5566603a67833a4ff97c8b1cd84010edbf63765d9681762b` | Adobe Setup.exe 18.9.20044.251705 |
| `51c263c86d5933248e4ba7b186eb12b5afa87c1a26be7dffc9ff1516efab9d0f` | AdobeCollabSync.exe 18.11.20055.290043 |
| `76f52cf20a71cb883c8c4048c14e3a88b6a5590a99a4f59c9d162f45e58c9471` | AcroBroker.exe 20.6.20034.366983 |
| `115e3d313832e8b316052d65848a8d236a17444c3d1c87818e3be0c4e96745fd` | no version info (7.2 MB) |
| `9ce158f5767ca6ed78677a1a3a64b481c9e3a5a3e763b777290e65290c275e5e` | Oracle javacpl.exe 11.241.2.07 |
| `93f1d015de77d727bb33081fef5ef0fc4098e256c554950bc539fd77cd73e349` | Oracle javaw.exe 8.0.2410.7 |
| `fdc0a130b156a7fdbae7d55896f245cec9db4d30ef8174a6531efa6f9b272705` | Oracle jp2launcher.exe 11.381.2.09 |
| `d051dfb9a5e492bf1999b193656a76091cc9946403c42dd2b941b30f66246987` | Mozilla maintenanceservice.exe 61.0.1 |
| `418871b8b598100a5f5e399e01a89422ca0a2a5b770fbc18cc58e352bb07a31c` | Mozilla crashreporter.exe 61.0.1 |
| `949f2bae70c44b36d0583d4f2c012a97aa3e2ba5e0c85be6302fcbbf64aab429` | Mozilla plugin-container.exe 61.0.1 |
| `795d8d2e4360bc1e39c67b02dae1601e4847169fc2f05cef261de1c0ccc64e61` | GoogleUpdateSetup.exe 1.3.36.152 |
| `4685a9c8f49f7d176caa00559f014aeb8b6cf6660d3c279f58dfadaea31700e7` | GoogleUpdateSetup.exe 1.3.36.152 |
| `649b448d5d617ecc4f55f999dede56386503249f4eabc64754ef8e0a013043e8` | nerds.de loopBeAudio.exe 1.2.0.0 |

### Memory images of infected processes

These two files are process-memory images of infected programs, probably dumped by a sandbox. The layer is already decrypted and the original entry bytes are restored.

| SHA256 | Host program |
|---|---|
| `2192360d42246a9ed475215fe7c6afed67d15ac80782bd52e44b8cdc74de77c0` | eFigureOut MouseLocator.EXE 1,0,0,1 |
| `c88c2d9c0d8059e60b1e9eb374b8d8c58c8077b10cb264b13ebb3ff0f78f0d66` | no version info |

## Analysis

The function names below are the malware's own export names. Addresses are RVAs in the v1 core (ImageBase `0x62400000`) unless marked otherwise. The example host is `a0c42a23` (ImageBase `0x400000`).

### Infection Chain

An infected program runs in this order:

1. The **entry stub** at the original entry point decodes a small transfer block and jumps to the layer decryptor.
2. The **layer decryptor** in the last section decrypts the layer and jumps to the loader.
3. The **loader** decrypts and maps the core DLL, then calls its `DllMain`.
4. `DllMain` starts the worm threads and copies the original entry bytes back.
5. The loader jumps to the original entry point and the host runs normally.

```
entry stub (host EP, .text, RWX)     last section (.reloc/.rsrc, RWX, enlarged)
+----------------------+             +------------------------------------------------------------------+
|     4-7 junk blocks  |  push/ret   | original section data | 100-200 junk blocks     | decryptor+key |
|     XOR-0x5F decoder | ----------> |   ...                 |  -> decrypt layer, jmp  | mult | count  |
|     transfer block   |             +-------------------------------------------------+---------------+
+----------------------+             | ENCRYPTED LAYER: loader junk (100-200 blocks)  | loader        |
                                     | (0x349 B) | API names | RC4(core DLL, UPX) 0x8C00 B | stolen bytes |
                                     | flag | METADATA BOX 0x28 B (rc4 key ... marker 20260715)         |
                                     +------------------------------------------------------------------+
```

The malware's log strings call these stages S0 (entry stub), S0Spec (transfer block), S1 (layer decryptor) and S2 (loader).

### Junk Code

Every stage is padded with generated junk blocks (`GenerateJunkBlock`, `0x2D31`). Each block:

- starts with `push 0FFF4F7xxh / pop r32` (bytes `68 xx F7 F4 FF 5x`), where `xx` is random;
- does some register arithmetic (`xchg`, `add [esp]`, `rol`/`ror`, `imul`, `movsx` ...);
- ends with an opaque branch that is always taken, a few dead bytes, and `xor r,r`.

The templates for each stage also get small instruction substitutions. For example `test ecx,ecx` becomes `or ecx,ecx`, and `add eax,13h` becomes `sub eax,-13h`. Infected hosts have 4-7 junk blocks in the entry stub and 100-200 in each of the layer decryptor and the loader.

### Entry Point Stub

`InfectFile` overwrites the bytes at the original entry point (OEP) with the entry stub and saves the original bytes inside the encrypted layer. The stub has three parts:

1. 4-7 junk blocks.
2. A XOR decoder (`xor byte [esi],5Fh` over N bytes, then `push edi / ret`). The key is always `0x5F` in v1.
3. The encoded transfer block. It computes the address of the layer decryptor and returns into it.

The decoder in host `a0c42a23`:

```
00415d00 e8e6ffffff   call 0x415ceb          ; pushes 0x415d05 (encoded transfer block)
00415ceb 5e           pop esi
00415cec 89f7         mov edi, esi
00415cee b93c000000   mov ecx, 0x3c          ; template has 0DEADBEEFh here
00415cf3 09c9         or ecx, ecx
00415cf5 7407         je 0x415cfe
00415cf7 80365f       xor byte ptr [esi], 0x5f
00415cfa 46           inc esi
00415cfb 49           dec ecx
00415cfc ebf5         jmp 0x415cf3
00415cfe 57           push edi
00415cff c3           ret                    ; -> decoded transfer block
```

The decoded transfer block adds a stored delta to its own address and returns to the layer decryptor:

```
00415d05 e800000000   call 0x415d0a
00415d0a 5b           pop ebx                ; ebx = 0x415d0a
00415d0b 0f6ec3       movd mm0, ebx          ; MMX round trip
00415d0e 0f7ec7       movd edi, mm0
00415d11 b8f6550500   mov eax, 0x555f6       ; delta
  ...                                        ; junk
00415d3a 01c7         add edi, eax           ; edi = 0x46b300 = layer decryptor entry
00415d3c 57           push edi
00415d40 c3           ret
```

In all 25 hosts the size of the entry stub matches the saved-bytes size in the metadata box, and the transfer block's target matches the layer decryptor entry in the box.

### Layer Decryptor

The layer decryptor starts with 100-200 junk blocks. Its core is a short decryptor followed by three dwords: the key, the multiplier and the byte count.

```
0046cc86 e800000000   call 0x46cc8b
0046cc8b 5d           pop ebp
0046cc8c 8d7d3c       lea edi, [ebp + 0x3c]
0046cc8f 8b0f         mov ecx, [edi]         ; count  (0xa335)
0046cc91 8b47f8       mov eax, [edi - 8]     ; key    (0x097b1a00)
0046cc94 8b57fc       mov edx, [edi - 4]     ; mult   (27)
0046cc97 8d7704       lea esi, [edi + 4]     ; layer start
0046cc9a 89f5         mov ebp, esi
0046cc9c 85c9         test ecx, ecx
0046cc9e 741d         je 0x46ccbd
0046cca0 0fafc2       imul eax, edx          ; a = a * mult
0046cca3 83e8ed       sub eax, -0x13         ; a = a + 0x13
0046cca6 89c3         mov ebx, eax
0046cca8 c1e303       shl ebx, 3
0046ccab c1e81d       shr eax, 0x1d
0046ccae 09d8         or eax, ebx            ; a = rol(a, 3)
0046ccb0 88c3         mov bl, al
0046ccb2 301e         xor byte ptr [esi], bl ; decrypt one byte
0046ccb4 0fb61e       movzx ebx, byte ptr [esi]
0046ccb7 01d8         add eax, ebx           ; feed back the plaintext byte
0046ccb9 46           inc esi
0046ccba 49           dec ecx
0046ccbb ebdf         jmp 0x46cc9c
0046ccbd ffe5         jmp ebp                ; -> loader
```

The cipher in short:

```
a = key
for each byte b:
    a = rol32(a * mult + 0x13, 3)
    b ^= a & 0xFF
    a += b          # plaintext byte
```

`InfectFile` picks a new key and multiplier for each infection. The key is `rand() << 16 ^ rand()`. The multiplier is `(rand() % 126) | 1`, an odd value from 1 to 125. `rand` is a standard LCG (`x * 0x41C64E6D + 0x3039`) seeded with `GetTickCount`.

### Loader

The loader is 0x349 bytes of position-independent code. It:

1. Saves all registers (`pushad / pushfd`).
2. Reads the host base from the PEB and finds kernel32 as the third entry in the module initialisation-order list.
3. Finds `GetProcAddress` by comparing the first 8 characters of each export name, then resolves `VirtualAlloc`, `Beep` and `LoadLibraryA`.
4. Plays beeps: 440 Hz, 660 Hz and 880 Hz.
5. Finds the metadata box at the end of the last section.
6. Copies the payload into new RWX memory and decrypts it with RC4 (8-byte key from the box).
7. Checks for `MZ`, maps the sections, applies relocations and resolves imports.
8. Beeps again (880 Hz, 1200 Hz) and calls the DLL entry point with `DLL_PROCESS_ATTACH`.
9. Restores registers and jumps to the original entry point.

The loader does not restore the stolen entry bytes. The core does that in `ExecuteTrampoline` (`0xCEA6`), which `DllMain` calls before it returns:

- it finds the box at the end of the host's last section;
- it checks that the stolen size is between 1 and `0x258`;
- it copies the saved bytes back to the OEP with `VirtualProtect`;
- it beeps once more at 1500 Hz.

In host `a0c42a23` the saved bytes disassemble to the original MSVC start-up code (`call __security_init_cookie / jmp __tmainCRTStartup`).

### Metadata Box

The last 0x28 bytes of the decrypted layer hold the infection metadata. Values are from host `a0c42a23`.

| Offset | Field | Example |
|---|---|---|
| +0x00 | RC4 key for the core (8 bytes, new per infection) | `291c58e31f3b7a7d` |
| +0x08 | Size of the saved entry bytes (= entry stub size, max 0x258) | `0x184` |
| +0x0C | OEP RVA | `0x15BBD` |
| +0x10 | Layer decryptor entry RVA | `0x6B300` |
| +0x14 | Payload offset inside the layer | `0x1575` |
| +0x18 | Layer RVA (= start of the loader) | `0x6CCCB` |
| +0x1C | Payload size (always `0x8C00`) | `0x8C00` |
| +0x20 | Payload RVA | `0x6E240` |
| +0x24 | Version marker | `0x20260715` |

Just before the box:

- `end - 0x30` is a flag. It is 0 in infected files and `0x5354` in the USB copy.
- `end - 0x2C` holds a value we could not identify.
- The saved entry bytes sit before that, aligned to 16 bytes.

When the worm infects the next file it reads its own RC4 key from this box, decrypts its own core, and re-encrypts it with a new key.

### What Gets Infected

`InfectFile` (`0x54FE`) accepts a file only if:

- it is a PE32 i386 EXE between 0x200 bytes and 40 MB;
- it is not .NET (the COM descriptor directory must be empty);
- the entry section is code or executable;
- it is not already infected with the current or a newer marker.

Overlay handling:

- An Authenticode signature overlay is accepted ("Overlay is Digital Signature. Safe to infect."). It is kept as overlay data, but the Security directory is zeroed, so Windows no longer sees a signature.
- An overlay larger than file size / 5, or larger than 512,000 bytes, is skipped ("Massive Overlay").
- An overlay that starts with a RAR, ZIP or 7z signature is skipped ("SFX Archive Signature").
- Any other overlay is backed up and restored.

`InfectFile` has three patch modes:

- `DIRECT_PATCH` writes the entry stub at the OEP. All 25 hosts use it.
- `STUB_PATCH` is used when the OEP is in the last section.
- An entry-point-obscuring mode searches for three byte patterns and writes an `E8` call. We have not seen a sample that uses it.

Header changes made to every infected file:

- the entry section and the last section are set to RWX;
- the last section grows to hold the layer decryptor and the layer;
- `DYNAMIC_BASE` and `NX_COMPAT` are cleared, and `RELOCS_STRIPPED` is set;
- the checksum and the Security directory are zeroed.

Our reading of why: clearing `DYNAMIC_BASE` keeps the host at its preferred base, so no relocation touches the patched entry code. Clearing `NX_COMPAT` lets the RWX code run.

`DisinfectFile` (`0x511C`) reverses an infection. It finds the layer decryptor, decrypts the layer, restores the entry bytes and truncates the file. The worm uses it to upgrade files that carry an older marker.

### Startup

On `DLL_PROCESS_ATTACH`, `DllMain` (`0xD1ED`):

1. turns off error dialogs and beeps at 750 Hz;
2. creates a private heap and an infection mutex;
3. if it can take the `@0826` mutex, starts five threads: `DownloadAndUpdate`, `CleanAllUsersTemp`, `ScanFile`, `CreateAutoRun` and `RegScanThread`;
4. otherwise starts `MonitorMutex`, which retries every 5 seconds and starts the threads when the mutex is free;
5. calls `ExecuteTrampoline` and returns, so the host program runs normally while the threads work.

### File Scanning

`ScanFile` (`0xB491`) runs in two phases:

- **Phase 1** starts one worker per removable or fixed drive from C: to Z: and builds a list of EXEs.
- **Phase 2** infects the files in the list, with a 2-second pause after each one.

A full cycle ends with a sleep of 500 seconds, or 5 seconds when nothing was found.

The scanner skips these folders: `$RECYCLE.BIN`, `System Volume Information`, `Sandbox`, `Program Data`, `Application Data`, `Roaming`, `Local Settings`, `Documents and Settings`, `Microsoft Shared`, `Documents`, `Microsoft`, `Locallow`, `Temp`, `Windows`. It also skips files named `sality.exe` and `bdagent.exe`.

The code logs "SKIPPING Program Files path", but the check only tests whether the path starts with `c:\aa`. This looks like a leftover test path. In practice Program Files is scanned, which fits the infected Adobe, Java, Firefox and Google Update files.

`RegScanThread` (`0x74B8`) reads the values under `HKCU\Software\Microsoft\Windows\CurrentVersion\Run` and infects each EXE they point to. Programs that start with Windows are infected first. This is the worm's only persistence. It never writes to the registry.

### USB and Drive Spreading

Every 5 seconds `CreateAutoRun` (`0xCBF2`) does the following for every removable or fixed drive:

1. Builds a new encrypted copy of the core with a new RC4 key.
2. Picks a file name. It re-uses the program named in an existing `autorun.inf`, or generates 4-6 random lowercase letters with `.exe` or `.pif`.
3. Writes a standalone EXE to the drive root and sets it read-only, hidden and system.
4. Writes a randomised `autorun.inf` and holds it open with an exclusive lock so it cannot be deleted.

The standalone EXE is built from scratch: one `.text` section holding the entry stub, the layer decryptor and the encrypted layer, with the box flag set to `0x5354`. When it runs, it opens Explorer on the drive root so the user sees the drive contents. It then keeps running for about 10 minutes while the threads infect files, and exits.

The `autorun.inf` contains `[AutoRun]` and six entries in random order: `open=`, `shellexecute=`, `shell\open\default=1`, `shell\open\command=`, `shell\explore\command=` and `shell\autoplay\command=`. The generator randomises letter case, spacing around `=`, comment lines and blank lines, so no two files are the same.

Windows 7 and later ignore `autorun.inf` on USB flash drives. XP and Vista got the same change in 2011 (KB971029). On current systems the USB copy only runs if someone opens it. Infected EXEs on removable drives still spread the worm without AutoRun.

### Update Channel

`DownloadAndUpdate` (`0x10C2`) connects to the update host on TCP port 8000. The protocol is plain TCP with NUL-terminated ASCII words:

1. Client sends `DOWNLOAD`.
2. Client waits 50 ms and sends `Intel(R) HD Graphics Manager.exe`.
3. Server sends a 4-byte little-endian length (1 byte to 10 MiB).
4. Client sends `READY`.
5. Server sends the file in chunks of up to 4,096 bytes.

The client writes the file to `%TEMP%` with 6 random letters and `.exe`, starts it with `CreateProcessW`, and the thread ends. If the download is incomplete, it deletes the file and tries again after 5 seconds. Socket timeouts are 30 seconds. There is no HTTP, no encryption and no check of the server or the file.

On 2026-11-17 `updatewin[.]ddns[.]net` resolved to `192.168.1[.]100`, a private LAN address, with a TTL of 60. `updatewinold[.]ddns[.]net` had no A record. We did not connect to either host, so we do not know what the server sends. A private address fits an author testing on a home or lab network. The name can be pointed at a public server at any time.

### Other Behaviour

- **AV installer sabotage.** If the file name contains `bitdef` or `kasp`, the patch writes eight `RET` (`0xC3`) bytes instead of a call, so the program returns at once. This is from the decompiled code only; we have no sample that shows it.
- **Temp cleaning.** Every 10 seconds `CleanAllUsersTemp` deletes `*.exe` and `*.rar` from every user's Temp folder. The code does not show why.
- **Beeps.** The loader, `DllMain` and `ExecuteTrampoline` beep at 440, 660, 880, 880, 1200, 750 and 1500 Hz.
- **Debug log.** Both builds write every log line to `D:\debug.log` when drive D: is writable. Together with the `c:\aa` test path, this looks like a development build.

### Vietnamese Log Strings

Most log strings are in Vietnamese with correct diacritics. This points to a Vietnamese-speaking author. It is a language observation, not attribution.

| Text | Translation |
|---|---|
| `[+] Đang khử nhiễm: %ls` | Disinfecting: %ls |
| `[+] Đã khôi phục OEP về 0x%X` | Restored the OEP to 0x%X |
| `[✔] Khử nhiễm thành công. File co rút về đúng vách nguyên bản gốc (0x%X).` | Disinfection successful. File shrunk back to its original boundary |
| `[+] File hoàn toàn sạch hoặc không chứa Metadata hợp lệ. Bỏ qua khâu dọn dẹp.` | File is clean or has no valid metadata. Skipping cleanup. |
| `[!] InfectFile: Phát hiện phiên bản cũ hơn. Cho phép trượt luồng xuống Bước 2 để khử nhiễm...` | Older version detected. Falling through to step 2 to disinfect... |
| `[ERROR] Khong tim thay 0xDEADBEEF trong s2mini_bin!` | 0xDEADBEEF not found in s2mini_bin! |

### Version 2

We have one v2 core (`0678e636`, a memory dump) and no v2-infected host. The main changes from v1:

- **Two update hosts.** It tries `updatewinold[.]ddns[.]net` first, then `updatewin[.]ddns[.]net`, both on TCP 8000.
- **No mutex.** `DllMain` starts one thread, `InitWorkerThread` (v2 `0xC404`), which starts the five worker threads without a single-instance check.
- **New payload cipher.** RC4 is replaced by a custom Feistel cipher on 8-byte blocks with 64 rounds per block. The key is the padded payload length, so the payload can be decrypted knowing only its size. The log says "60000 rounds", which does not match the code.
- **New entry stub decoder.** A position-keyed byte stream: `k(i) = ror32(rol32((i ^ 0x9E3779B9) * 0x045D9F3B, 7) + i, 13)`.
- **New layer decryptor lookup.** The layer decryptor searches for the 3-byte marker `M^4` (`4D 5E 34`) and reads the key, multiplier and count after it. The layer cipher is unchanged.
- **New loader.** It starts with the marker `S2EN`, resolves `GetProcAddress` by ROR-13 hash `0x7C0DFCAA`, and contains an assembly copy of the Feistel decryptor.

### Decoding the Config

The script below decrypts the layer of an infected v1 host, parses the metadata box and decrypts the packed core DLL. It also handles the memory images, where the layer is already decrypted. It needs `pefile`, and `upx` for the second step.

In [1]:
import hashlib
import re
import struct

import pefile

# Layer decryptor head: call $+5 / pop ebp / lea edi,[ebp+3Ch] / mov ecx,[edi] / mov eax,[edi-8] / mov edx,[edi-4]
S1_HEAD = bytes.fromhex('e8000000005d8d7d3c8b0f8b47f88b57fc')

# Metadata box: last 0x28 bytes of the decrypted layer
BOX = struct.Struct('<8s8I')
BOX_FIELDS = ('rc4_key', 'stolen_size', 'oep_rva', 'stage1_rva', 'payload_off',
              'layer_rva', 'payload_size', 'payload_rva', 'marker')


def rol32(x, n):
    return ((x << n) | (x >> (32 - n))) & 0xFFFFFFFF


def s1_decrypt(data, seed, mult):
    """Layer decryptor cipher: a = rol32(a*mult + 0x13, 3); byte ^= a & 0xFF; a += plaintext byte."""
    out, a = bytearray(data), seed
    for i, c in enumerate(out):
        a = rol32((a * mult + 0x13) & 0xFFFFFFFF, 3)
        out[i] = c ^ (a & 0xFF)
        a = (a + out[i]) & 0xFFFFFFFF
    return bytes(out)


def rc4(key, data):
    S, j = list(range(256)), 0
    for i in range(256):
        j = (j + S[i] + key[i % len(key)]) & 0xFF
        S[i], S[j] = S[j], S[i]
    out, i, j = bytearray(), 0, 0
    for c in data:
        i = (i + 1) & 0xFF
        j = (j + S[i]) & 0xFF
        S[i], S[j] = S[j], S[i]
        out.append(c ^ S[(S[i] + S[j]) & 0xFF])
    return bytes(out)


def decode_host(data):
    """Decrypt the layer of an infected v1 host and return (config, packed core DLL)."""
    pe = pefile.PE(data=data, fast_load=True)
    last = pe.sections[-1]
    sec = data[last.PointerToRawData:last.PointerToRawData + last.SizeOfRawData]
    head = sec.find(S1_HEAD)
    if head < 0:
        return None, None
    kb = head + 5 + 0x3C                       # edi = (address of pop ebp) + 3Ch
    seed, mult, count = struct.unpack_from('<III', sec, kb - 8)
    enc = sec[kb + 4:kb + 4 + count]
    # Infected files hold the layer encrypted. Process-memory images hold it already decrypted.
    for form, layer in (('infected file', s1_decrypt(enc, seed, mult)), ('memory image', enc)):
        box = dict(zip(BOX_FIELDS, BOX.unpack_from(layer, len(layer) - BOX.size)))
        if box['marker'] >> 24 != 0x20:
            continue
        core = rc4(box['rc4_key'], layer[box['payload_off']:box['payload_off'] + box['payload_size']])
        if core[:2] != b'MZ':
            continue
        config = {
            'form': form,
            'layer_seed': hex(seed),
            'layer_mult': mult,
            'rc4_key': box['rc4_key'].hex(),
            'oep_rva': hex(box['oep_rva']),
            'stolen_size': box['stolen_size'],
            'marker': hex(box['marker']),
            'core_sha256': hashlib.sha256(core).hexdigest(),
        }
        return config, core
    return None, None


def core_strings(core):
    """Pull the update hosts and version banner from an unpacked core DLL."""
    return {
        'update_hosts': sorted({m.decode() for m in re.findall(rb'[a-z0-9.-]+\.ddns\.net', core)}),
        'version': [m.decode() for m in re.findall(rb'VERSION (\d\d/\d\d/\d{4})', core)],
        'request_name': [m.decode() for m in re.findall(rb'Intel\(R\)[ -~]+?\.exe', core)],
    }

Decrypt the layer of host `a0c42a23` and print the metadata:

In [2]:
SAMPLE = 'a0c42a23fbcd3cd8c21db68517e18b5c229e9a2d95af62a5a57407ff92e48141'

with open(SAMPLE, 'rb') as f:
    config, core = decode_host(f.read())

for k, v in config.items():
    print(f'{k:12} {v}')

form         infected file
layer_seed   0x97b1a00
layer_mult   27
rc4_key      291c58e31f3b7a7d
oep_rva      0x15bbd
stolen_size  388
marker       0x20260715
core_sha256  ec55c4ba2ab0b530c4d613e9c0214611c734fc0d17e2fabec68ade33b980075c


The core is UPX packed, so the update host is not visible yet. Unpack it with `upx -d` and pull the strings:

In [3]:
import os
import subprocess
import tempfile

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'core.dll')
    with open(path, 'wb') as f:
        f.write(core)
    subprocess.run(['upx', '-q', '-d', path], check=True, capture_output=True)
    with open(path, 'rb') as f:
        unpacked = f.read()

info = core_strings(unpacked)
info['update_hosts'] = [h.replace('.', '[.]') for h in info['update_hosts']]
for k, v in info.items():
    print(f'{k:12} {v}')

update_hosts ['updatewin[.]ddns[.]net']
version      ['15/07/2026']
request_name ['Intel(R) HD Graphics Manager.exe']


The v2 core we have is a memory dump with its code in clear, so `core_strings` works on it directly:

In [4]:
with open('0678e636166df303548c8c2f4d75611304f7f01d5881ac6e4baad16dae0aa566', 'rb') as f:
    info = core_strings(f.read())

info['update_hosts'] = [h.replace('.', '[.]') for h in info['update_hosts']]
for k, v in info.items():
    print(f'{k:12} {v}')

update_hosts ['updatewin[.]ddns[.]net', 'updatewinold[.]ddns[.]net']
version      ['23/08/2026']
request_name ['Intel(R) HD Graphics Manager.exe']


## IOCs

### Network

| Indicator | Notes |
|---|---|
| `updatewin[.]ddns[.]net:8000` | v1 update host; v2 second choice |
| `updatewinold[.]ddns[.]net:8000` | v2 first choice |

Protocol strings sent by the client: `DOWNLOAD`, `Intel(R) HD Graphics Manager.exe`, `READY`.

### Host

| Artefact | Value |
|---|---|
| Downloaded update | `%TEMP%\[a-z]{6}.exe` |
| Drive-root copy | `X:\[a-z]{4,6}.exe` or `.pif`, read-only + hidden + system |
| Autorun file | `X:\autorun.inf` (old copy renamed `autorun.inf.old` if it cannot be deleted) |
| Debug log | `D:\debug.log` |
| Mutex (v1) | `@0826` |
| Registry (read only) | `HKCU\Software\Microsoft\Windows\CurrentVersion\Run` |

### Infected File Traits

- The entry section has characteristics `0xE0000020` (RWX).
- The last section (`.reloc` or `.rsrc`) is enlarged and has characteristics `0xE2000020` or `0xE0000020`.
- `DllCharacteristics` has `DYNAMIC_BASE` and `NX_COMPAT` cleared. `RELOCS_STRIPPED` is set.
- The checksum is 0 and the Security directory is zeroed, but the old certificate bytes remain as overlay.
- The entry point is a junk stub. The v1 entry stub decoder `74 07 80 36 5F 46 49 EB F5 57 C3` is 0x100-0x160 bytes after it.
- The v1 layer decryptor head `E8 00 00 00 00 5D 8D 7D 3C 8B 0F 8B 47 F8 8B 57 FC 8D 77 04 89 F5` is in the last section.
- Junk blocks start with `68 ?? F7 F4 FF 5?`.